In [6]:
class FunctionWrapper:
    def __init__(self, func, *default_args, **default_kwargs):
        self.func = func
        # I default per gli args e kwargs vengono memorizzati
        self.default_args = default_args
        self.default_kwargs = default_kwargs

    def __call__(self, *args, **kwargs):
        # Se l'utente non fornisce args, si usano i default
        if not args:
            final_args = self.default_args
        else:
            # Se l'utente fornisce args, questi sovrascrivono i default
            final_args = args

        # Per i kwargs si fa un merge: si parte dai default e si aggiorna con quelli dell'utente
        final_kwargs = self.default_kwargs
        final_kwargs.update(kwargs)

        # Invocazione della funzione wrappata con i parametri finali
        return self.func(*final_args, **final_kwargs)


# Esempio d'uso


def my_function(a, b=2, c=3):
    return a + b + c


# Wrappiamo la funzione impostando default_args e default_kwargs
# Qui impostiamo a=1 come default, b=2 e c=5 come default dei kwargs
wrapped = FunctionWrapper(my_function, 1, b=2, c=5)

print(wrapped())  # Usa default: a=1, b=2, c=5 -> 1+2+5 = 8
print(wrapped(10))  # Sovrascrive gli args: a=10, usa default: b=2, c=5 -> 17
print(wrapped(c=10))  # Usa default args: a=1, b=2, sovrascrive c=10 -> 1+2+10=13
print(wrapped(2, c=1))  # Sovrascrive args: a=2, b=2, e c=1 da kwargs -> 2+2+1=5


8
17
13
5


In [7]:
import numpy as np
import re
from astropy.modeling.models import Gaussian1D
import timeit

astropy_gaussian = Gaussian1D()

def gaussian(x, mu, sigma=2):
    return (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

wrapped = FunctionWrapper(gaussian, 1,2,  sigma=2)


def eval():
    return wrapped(2,3)


#def call():
#    return model(0, mu=11)


def astropy_mod():
    return astropy_gaussian(0)


time_original = timeit.timeit(eval, number=100000)
#time_optimized = timeit.timeit(call, number=100000)
time_astro = timeit.timeit(astropy_mod, number=100000)


print(f"Tempo funzione eval: {time_original:.6f} secondi")
#print(f"Tempo funzione __call__: {time_optimized:.6f} secondi")
print(f"Tempo funzione astropy: {time_astro:.6f} secondi")


Tempo funzione eval: 0.363645 secondi
Tempo funzione astropy: 4.790893 secondi
